In [2]:
import fastf1
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import os


if not os.path.exists('f1_cache'):
    os.makedirs('f1_cache')

# Now enable cache
fastf1.Cache.enable_cache('f1_cache')


Goal: Predict winner of 2025 Australian Grand Prix

In [3]:
race_2024 = fastf1.get_session(2024, "Australia", "R")
race_2024.load()

qualify_2025 = fastf1.get_session(2025, "Australia", "Q")
qualify_2025.load()

print("Both sessions loaded")

core           INFO 	Loading data for Australian Grand Prix - Race [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 19 drivers: ['55', '16', '4', '81', '11', '18', '22', '14', '27', '20', '23', '3', '10', '77', '24', '31', '63', '44', '1']
core           INFO 	Loading data for Australian Grand Prix - Qu

Both sessions loaded


In [4]:
print("2025 Quali columns:")
print(qualify_2025.results.columns.tolist())

# 2024 Race results
print("\n2024 Race columns:")
print(race_2024.results.columns.tolist())

2025 Quali columns:
['DriverNumber', 'BroadcastName', 'Abbreviation', 'DriverId', 'TeamName', 'TeamColor', 'TeamId', 'FirstName', 'LastName', 'FullName', 'HeadshotUrl', 'CountryCode', 'Position', 'ClassifiedPosition', 'GridPosition', 'Q1', 'Q2', 'Q3', 'Time', 'Status', 'Points', 'Laps']

2024 Race columns:
['DriverNumber', 'BroadcastName', 'Abbreviation', 'DriverId', 'TeamName', 'TeamColor', 'TeamId', 'FirstName', 'LastName', 'FullName', 'HeadshotUrl', 'CountryCode', 'Position', 'ClassifiedPosition', 'GridPosition', 'Q1', 'Q2', 'Q3', 'Time', 'Status', 'Points', 'Laps']


In [30]:
def fill_quali_times(quali_result):
    df  =  quali_result.copy()

    df["Q1_seconds"] = df["Q1"].dt.total_seconds()
    df["Q2_seconds"] = df["Q2"].dt.total_seconds()
    df["Q3_seconds"] = df["Q3"].dt.total_seconds()

    # Get the slowest Q2,Q3 time
    slowest_q1 = df["Q1_seconds"].max()
    slowest_q2 = df["Q2_seconds"].max()
    slowest_q3 = df['Q3_seconds'].max()

    # Fill missing Q2 (positions 11-20)
    # Add 0.1 seconds penalty for each position behind P10
    for idx, row in df.iterrows():
        
        if pd.isna(row['Q1_seconds']):
            penalty = position * 0.1 
            df.loc[idx, 'Q1_seconds'] = slowest_q1 + penalty

        if pd.isna(row["Q2_seconds"]):
            position= row["Position"]
            penalty = (position - 10) * 0.1
            df.loc[idx, "Q2_seconds"] = slowest_q2 + penalty

        if pd.isna(row['Q3_seconds']):
            position = row['Position']
            penalty = (position - 10) * 0.1
            df.loc[idx, 'Q3_seconds'] = slowest_q3 + penalty


    return df


quali_2025_filled = fill_quali_times(qualify_2025.results)
# print(quali_2025_filled[['Abbreviation', 'Position', 'Q1_seconds', 'Q2_seconds', 'Q3_seconds']])

# Convert to DataFrame
quali_2025_df = pd.DataFrame(quali_2025_filled)

# Get 2024 race results
race_2024_df = pd.DataFrame(race_2024.results)

#  Create a mapping: driver -> their 2024 position
driver_2024_position = race_2024_df.set_index('Abbreviation')['Position'].to_dict()
driver_2024_position

{'SAI': 1.0,
 'LEC': 2.0,
 'NOR': 3.0,
 'PIA': 4.0,
 'PER': 5.0,
 'STR': 6.0,
 'TSU': 7.0,
 'ALO': 8.0,
 'HUL': 9.0,
 'MAG': 10.0,
 'ALB': 11.0,
 'RIC': 12.0,
 'GAS': 13.0,
 'BOT': 14.0,
 'ZHO': 15.0,
 'OCO': 16.0,
 'RUS': 17.0,
 'HAM': 18.0,
 'VER': 19.0}

In [26]:
print(race_2024.results.Position)

55     1.0
16     2.0
4      3.0
81     4.0
11     5.0
18     6.0
22     7.0
14     8.0
27     9.0
20    10.0
23    11.0
3     12.0
10    13.0
77    14.0
24    15.0
31    16.0
63    17.0
44    18.0
1     19.0
Name: Position, dtype: float64
